In [ ]:
import os, random, warnings, hashlib, pickle
import numpy as np
import pandas as pd
import librosa
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV

warnings.filterwarnings("ignore")

# ── Try LightGBM; fall back to sklearn GradientBoosting ──────────────────────
try:
    import lightgbm as lgb
    USE_LGBM = True
    print("LightGBM available ✓")
except ImportError:
    from sklearn.ensemble import GradientBoostingClassifier, VotingClassifier
    from sklearn.svm import SVC
    USE_LGBM = False
    print("LightGBM not found — falling back to sklearn ensemble")

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────
SEED = 42
SR = 22050
DURATION = 5.0          # seconds per segment
NUM_SAMPLES = int(SR * DURATION)
N_MFCC = 40           # MFCC coefficients
N_SEGMENTS = 6            # segments per track during training (data augmentation)
N_INFER_SEG = 10           # segments for test-time ensemble

DATA_ROOT = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems'
MASHUP_ROOT = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/mashups'
TEST_CSV = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv'
TRAIN_MASHUP_CSV = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/train.csv'
CACHE_DIR = '/kaggle/working/feat_cache'
SAVE_PATH = '/kaggle/working/lgbm_models.pkl'

STEMS = ['vocals.wav', 'drums.wav', 'bass.wav', 'other.wav']
GENRES = ['blues','classical','country','disco','hiphop',
          'jazz','metal','pop','reggae','rock']

random.seed(SEED)
np.random.seed(SEED)
os.makedirs(CACHE_DIR, exist_ok=True)


# ─────────────────────────────────────────────────────────────────────────────
# FEATURE EXTRACTION
# ─────────────────────────────────────────────────────────────────────────────

def extract_features(y: np.ndarray, sr: int = SR) -> np.ndarray:
    # Ensure mono float32
    if y.ndim > 1:
        y = y.mean(axis=0)
    y = y.astype(np.float32)

    feats = []

    # MFCC + delta
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC)
    dmfcc = librosa.feature.delta(mfcc)
    feats += [mfcc.mean(1), mfcc.std(1), dmfcc.mean(1), dmfcc.std(1)]

    # Spectral contrast (captures harmonic vs percussive energy per band)
    sc = librosa.feature.spectral_contrast(y=y, sr=sr)
    feats += [sc.mean(1), sc.std(1)]

    # Zero-crossing rate
    zcr = librosa.feature.zero_crossing_rate(y)
    feats += [zcr.mean(keepdims=True).flatten(), zcr.std(keepdims=True).flatten()]

    # RMS energy
    rms = librosa.feature.rms(y=y)
    feats += [rms.mean(keepdims=True).flatten(), rms.std(keepdims=True).flatten()]

    # Spectral centroid
    cent = librosa.feature.spectral_centroid(y=y, sr=sr)
    feats += [cent.mean(keepdims=True).flatten(), cent.std(keepdims=True).flatten()]

    # Spectral rolloff
    roll = librosa.feature.spectral_rolloff(y=y, sr=sr)
    feats += [roll.mean(keepdims=True).flatten(), roll.std(keepdims=True).flatten()]

    # Spectral bandwidth
    bw = librosa.feature.spectral_bandwidth(y=y, sr=sr)
    feats += [bw.mean(keepdims=True).flatten(), bw.std(keepdims=True).flatten()]

    # Tempo (BPM) — single scalar, very genre-discriminative
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    feats.append(np.array([float(tempo)]))

    return np.concatenate(feats).astype(np.float32)


def load_audio_segment(path: str, start_sample: int = 0) -> np.ndarray:
    """Load a DURATION-second chunk starting at start_sample."""
    y, _ = librosa.load(path, sr=SR, offset=start_sample / SR,
                        duration=DURATION, mono=True)
    if len(y) < NUM_SAMPLES:
        y = np.pad(y, (0, NUM_SAMPLES - len(y)))
    else:
        y = y[:NUM_SAMPLES]
    return y


def cache_key(path: str, start: int) -> str:
    h = hashlib.md5(f"{path}:{start}".encode()).hexdigest()[:16]
    return os.path.join(CACHE_DIR, h + '.npy')


def get_features_cached(path: str, start: int = 0) -> np.ndarray:
    """Extract features with disk caching."""
    ck = cache_key(path, start)
    if os.path.exists(ck):
        return np.load(ck)
    y = load_audio_segment(path, start)
    feat = extract_features(y)
    np.save(ck, feat)
    return feat


def get_file_duration_samples(path: str) -> int:
    """Return total samples in audio file without loading all data."""
    info = librosa.get_duration(path=path)
    return int(info * SR)


def multi_segment_features(path: str, n_segments: int) -> np.ndarray:
    total = get_file_duration_samples(path)
    if total <= NUM_SAMPLES:
        starts = [0] * n_segments
    else:
        step = (total - NUM_SAMPLES) // max(1, n_segments - 1)
        starts = [min(i * step, total - NUM_SAMPLES) for i in range(n_segments)]

    return np.stack([get_features_cached(path, s) for s in starts])


# ─────────────────────────────────────────────────────────────────────────────
# BUILD TRAINING DATA
# ─────────────────────────────────────────────────────────────────────────────

def build_training_data():
    X_list, y_list, src_list = [], [], []

    # ── A: Stem files ─────────────────────────────────────────────────────────
    print("\n[1/2] Extracting features from stems …")
    stem_count = 0
    for genre in GENRES:
        genre_dir = os.path.join(DATA_ROOT, genre)
        if not os.path.exists(genre_dir):
            print(f"  WARNING: {genre_dir} missing")
            continue
        label = GENRES.index(genre)
        for track in sorted(os.listdir(genre_dir)):
            track_dir = os.path.join(genre_dir, track)
            if not os.path.isdir(track_dir):
                continue
            for stem in STEMS:
                spath = os.path.join(track_dir, stem)
                if not os.path.exists(spath):
                    continue
                total = get_file_duration_samples(spath)
                if total <= 0:
                    continue
                # Sample N_SEGMENTS evenly + random jitter for augmentation
                if total <= NUM_SAMPLES:
                    starts = [0] * N_SEGMENTS
                else:
                    base_step = (total - NUM_SAMPLES) // (N_SEGMENTS - 1)
                    starts = []
                    for i in range(N_SEGMENTS):
                        s = i * base_step
                        # ±0.5s random jitter
                        jitter = random.randint(-SR // 2, SR // 2)
                        s = max(0, min(total - NUM_SAMPLES, s + jitter))
                        starts.append(s)

                for start in starts:
                    feat = get_features_cached(spath, start)
                    X_list.append(feat)
                    y_list.append(label)
                    src_list.append('stem')
                stem_count += 1

        print(f"  {genre}: done")

    print(f"  Total stems processed: {stem_count}")

    # ── B: Mashup training data (closes domain gap) ───────────────────────────
    print("\n[2/2] Looking for mashup training data …")
    mashup_count = 0

    # Try train.csv first
    if os.path.exists(TRAIN_MASHUP_CSV):
        train_mdf = pd.read_csv(TRAIN_MASHUP_CSV)
        print(f"  Found train.csv with {len(train_mdf)} mashup rows")
        for _, row in train_mdf.iterrows():
            genre = row.get('genre', row.get('label', None))
            fname = row.get('filename', row.get('file', None))
            if genre is None or fname is None or genre not in GENRES:
                continue
            fpath = os.path.join(MASHUP_ROOT, fname)
            if not os.path.exists(fpath):
                continue
            label = GENRES.index(genre)
            feats = multi_segment_features(fpath, N_SEGMENTS)
            for feat in feats:
                X_list.append(feat)
                y_list.append(label)
                src_list.append('mashup_train')
            mashup_count += 1
        print(f"  Loaded {mashup_count} mashup training files")
    else:
        # Fall back: scan mashups/ for subfolders named by genre
        for genre in GENRES:
            gdir = os.path.join(MASHUP_ROOT, genre)
            if not os.path.isdir(gdir):
                continue
            label = GENRES.index(genre)
            for fname in sorted(os.listdir(gdir)):
                fpath = os.path.join(gdir, fname)
                if not os.path.isfile(fpath):
                    continue
                feats = multi_segment_features(fpath, N_SEGMENTS)
                for feat in feats:
                    X_list.append(feat)
                    y_list.append(label)
                    src_list.append('mashup_train')
                mashup_count += 1
        print(f"  Loaded {mashup_count} mashup files from genre subfolders")

    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.int32)
    print(f"\nDataset: {X.shape[0]} samples × {X.shape[1]} features")
    print(f"  stems={sum(s=='stem' for s in src_list)}, "
          f"mashup={sum('mashup' in s for s in src_list)}")
    for i, g in enumerate(GENRES):
        print(f"  {g:12s}: {(y==i).sum()}")

    return X, y


# ─────────────────────────────────────────────────────────────────────────────
# MODEL — LightGBM with 5-fold CV
# ─────────────────────────────────────────────────────────────────────────────

def train_lgbm(X: np.ndarray, y: np.ndarray):
    scaler = StandardScaler()
    X_sc = scaler.fit_transform(X)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    models = []
    oof_preds = np.zeros((len(y), len(GENRES)), dtype=np.float32)

    lgbm_params = dict(
        objective = 'multiclass',
        num_class = len(GENRES),
        metric = 'multi_logloss',
        n_estimators = 800,
        learning_rate = 0.05,
        num_leaves = 63,
        max_depth = -1,
        min_child_samples= 10,
        subsample = 0.8,
        subsample_freq = 1,
        colsample_bytree = 0.8,
        reg_alpha = 0.1,
        reg_lambda = 0.1,
        class_weight = 'balanced',   # handles class imbalance → better macro-F1
        n_jobs = -1,           # use all CPU cores
        random_state = SEED,
        verbose = -1,
    )

    print(f"\nTraining LightGBM with 5-fold stratified CV …")
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_sc, y)):
        X_tr, X_va = X_sc[tr_idx], X_sc[va_idx]
        y_tr, y_va = y[tr_idx],    y[va_idx]

        clf = lgb.LGBMClassifier(**lgbm_params)
        clf.fit(
            X_tr, y_tr,
            eval_set = [(X_va, y_va)],
            callbacks = [
                lgb.early_stopping(50, verbose=False),
                lgb.log_evaluation(period=100),
            ],
        )

        proba = clf.predict_proba(X_va)
        oof_preds[va_idx] = proba
        f1 = f1_score(y_va, proba.argmax(1), average='macro')
        print(f"  Fold {fold+1}/5 | Macro-F1: {f1:.4f} | "
              f"Best iter: {clf.best_iteration_}")
        models.append(clf)

    oof_f1 = f1_score(y, oof_preds.argmax(1), average='macro')
    print(f"\nOOF Macro-F1: {oof_f1:.4f}")
    print(classification_report(y, oof_preds.argmax(1), target_names=GENRES))
    return models, scaler


def train_sklearn_ensemble(X: np.ndarray, y: np.ndarray):
    scaler = StandardScaler()
    X_sc = scaler.fit_transform(X)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    oof_preds = np.zeros((len(y), len(GENRES)), dtype=np.float32)
    models = []

    print("\nTraining sklearn ensemble (RF + ExtraTrees + SVM) with 5-fold CV …")
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_sc, y)):
        X_tr, X_va = X_sc[tr_idx], X_sc[va_idx]
        y_tr, y_va = y[tr_idx],    y[va_idx]

        rf = RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                     n_jobs=-1, random_state=SEED)
        et = ExtraTreesClassifier(n_estimators=300, class_weight='balanced',
                                   n_jobs=-1, random_state=SEED)
        svm = CalibratedClassifierCV(
                SVC(kernel='rbf', C=10, class_weight='balanced', random_state=SEED),
                cv=3)

        rf.fit(X_tr, y_tr);  et.fit(X_tr, y_tr);  svm.fit(X_tr, y_tr)

        proba = (rf.predict_proba(X_va) +
                 et.predict_proba(X_va) +
                 svm.predict_proba(X_va)) / 3.0
        oof_preds[va_idx] = proba
        f1 = f1_score(y_va, proba.argmax(1), average='macro')
        print(f"  Fold {fold+1}/5 | Macro-F1: {f1:.4f}")
        models.append((rf, et, svm))

    oof_f1 = f1_score(y, oof_preds.argmax(1), average='macro')
    print(f"\nOOF Macro-F1: {oof_f1:.4f}")
    print(classification_report(y, oof_preds.argmax(1), target_names=GENRES))
    return models, scaler


# ─────────────────────────────────────────────────────────────────────────────
# INFERENCE
# ─────────────────────────────────────────────────────────────────────────────

def predict_mashup(path: str, models, scaler, n_segments: int = N_INFER_SEG) -> str:
    total = get_file_duration_samples(path)
    if total <= NUM_SAMPLES:
        starts = [0] * n_segments
    else:
        step = (total - NUM_SAMPLES) // max(1, n_segments - 1)
        starts = [min(i * step, total - NUM_SAMPLES) for i in range(n_segments)]

    feats = np.stack([get_features_cached(path, s) for s in starts])  # (S, D)
    feats_sc = scaler.transform(feats)                                 # (S, D)

    if USE_LGBM:
        # Average over fold models and segments
        prob_sum = np.zeros((feats_sc.shape[0], len(GENRES)), dtype=np.float64)
        for clf in models:
            prob_sum += clf.predict_proba(feats_sc)
        proba = prob_sum / len(models)              # (S, C)
    else:
        prob_sum = np.zeros((feats_sc.shape[0], len(GENRES)), dtype=np.float64)
        for (rf, et, svm) in models:
            prob_sum += (rf.predict_proba(feats_sc) +
                         et.predict_proba(feats_sc) +
                         svm.predict_proba(feats_sc)) / 3.0
        proba = prob_sum / len(models)

    # Average over segments → final prediction
    mean_proba = proba.mean(axis=0)
    return GENRES[mean_proba.argmax()]


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────

def main():
    # ── 1. Build features ────────────────────────────────────────────────────
    X, y = build_training_data()

    if len(X) == 0:
        raise RuntimeError("No training samples found. Check DATA_ROOT paths.")

    # ── 2. Train ─────────────────────────────────────────────────────────────
    if USE_LGBM:
        models, scaler = train_lgbm(X, y)
    else:
        models, scaler = train_sklearn_ensemble(X, y)

    # Save models
    with open(SAVE_PATH, 'wb') as f:
        pickle.dump({'models': models, 'scaler': scaler,
                     'use_lgbm': USE_LGBM}, f)
    print(f"\nModels saved → {SAVE_PATH}")

    # ── 3. Generate submission ────────────────────────────────────────────────
    test_df = pd.read_csv(TEST_CSV)
    print(f"\nGenerating predictions for {len(test_df)} test files …")

    predictions = []
    for i, (_, row) in enumerate(test_df.iterrows()):
        fpath = os.path.join(MASHUP_ROOT, row['filename'])
        if not os.path.exists(fpath):
            print(f"  MISSING: {fpath} → defaulting to 'blues'")
            predictions.append('blues')
            continue
        pred = predict_mashup(fpath, models, scaler)
        predictions.append(pred)
        if (i + 1) % 10 == 0:
            print(f"  {i+1}/{len(test_df)} done")

    test_df['genre'] = predictions
    out_path = '/kaggle/working/submission.csv'
    test_df[['id', 'genre']].to_csv(out_path, index=False)
    print(f"\nSubmission saved → {out_path}")

    # Distribution sanity check
    print("\nPrediction distribution:")
    for g, cnt in pd.Series(predictions).value_counts().items():
        print(f"  {g:12s}: {cnt}")


if __name__ == '__main__':
    main()